In [35]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


# Imports

In [36]:
import tensorflow as tf
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import itertools
from itertools import product
import xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Load Datasets

In [37]:
train_df = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# Prepare Datasets

In [38]:
def preprocess(df):
    df = df.copy()

    # ---- Name features ----
    def extract_title(name):
        title = name.split(",")[1].split(".")[0].strip()
        return title

    df["Title"] = df["Name"].apply(extract_title)

    rare_titles = [
        "Lady", "Countess", "Capt", "Col", "Don",
        "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"
    ]
    df["Title"] = df["Title"].replace(rare_titles, "Rare")
    df = df.drop("Name", axis=1)

    # ---- Ticket features ----
    def ticket_number(x):
        value = x.split(" ")[-1]
        return int(value) if value.isdigit() else -1

    def ticket_item(x):
        items = x.split(" ")
        if len(items) == 1:
            return "NONE"
        return "_".join(items[:-1])

    df["Ticket_number"] = df["Ticket"].apply(ticket_number)
    df["Ticket_item"] = df["Ticket"].apply(ticket_item)
    df = df.drop("Ticket", axis=1)

    # ---- Cabin features ----
    df["Deck"] = df["Cabin"].str[0]
    df["HasCabin"] = df["Cabin"].notna().astype(int)
    df["Deck"] = df["Deck"].fillna("Unknown")   # <-- explicit fill, was implicit NaN before
    df = df.drop("Cabin", axis=1)

    # ---- Family features ----
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    # ---- Fare feature ----
    df["FarePerPerson"] = df["Fare"] / df["FamilySize"]

    return df

train_df = preprocess(train_df)
test_df = preprocess(test_df)


train_df["Age"] = train_df.groupby("Title")["Age"].transform(lambda x: x.fillna(x.median()))
test_df["Age"] = test_df.groupby("Title")["Age"].transform(lambda x: x.fillna(x.median()))
test_df["Fare"] = test_df.groupby("Pclass")["Fare"].transform(lambda x: x.fillna(x.median()))

ticket_counts = train_df["Ticket_item"].value_counts()
rare = ticket_counts[ticket_counts < 5].index
train_df["Ticket_item"] = train_df["Ticket_item"].replace(rare, "Rare")
test_df["Ticket_item"] = test_df["Ticket_item"].apply(lambda x: "Rare" if x not in ticket_counts.index or x in rare else x)
train_df.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,Ticket_number,Ticket_item,Deck,HasCabin,FamilySize,IsAlone,FarePerPerson
0,1,0,3,male,22.0,1,0,7.2500,S,Mr,21171,A/5,Unknown,0,2,0,3.62500
1,2,1,1,female,38.0,1,0,71.2833,C,Mrs,17599,PC,C,1,2,0,35.64165
2,3,1,3,female,26.0,0,0,7.9250,S,Miss,3101282,STON/O2.,Unknown,0,1,1,7.92500
3,4,1,1,female,35.0,1,0,53.1000,S,Mrs,113803,NONE,C,1,2,0,26.55000
4,5,0,3,male,35.0,0,0,8.0500,S,Mr,373450,NONE,Unknown,0,1,1,8.05000


# Ensemble models

In [39]:
input_features = [
    "Pclass",
    "Sex",
    "Age",
    "Fare",
    "Embarked",

    # Name information
    "Title",

    # Family information
    "FamilySize",
    "IsAlone",

    # Cabin information
    "Deck",
    "HasCabin",

    # Ticket group information
    "Ticket_item",

    # engineered features
    "FarePerPerson",
]


train_df = train_df[input_features + ["Survived"]]
test_df = test_df[input_features + ["PassengerId"]]

X_train = train_df.drop("Survived", axis=1)
y_train = train_df["Survived"]

X_test = test_df.drop("PassengerId", axis=1)

X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

## Random Forest

In [41]:
param_dist = {
    "n_estimators": [300, 500, 1000, 1500, 2000],
    "max_depth": [None, 10, 15, 20, 25],
    "min_samples_leaf": [1, 2, 4, 8],
    "min_samples_split": [2, 5, 10, 15, 20],
    "max_features": ["sqrt", "log2", None]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=150,
    scoring="accuracy",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)
rf_best_model = rf_search.best_estimator_

print("===== BEST RF MODEL =====")
print(rf_search.best_params_)
print("5-fold CV accuracy:", rf_search.best_score_)

Fitting 5 folds for each of 150 candidates, totalling 750 fits
===== BEST RF MODEL =====
{'n_estimators': 1500, 'min_samples_split': 10, 'min_samples_leaf': 8, 'max_features': None, 'max_depth': 15}
5-fold CV accuracy: 0.84509446990145


## Boosting

In [42]:
param_dist = {
    "max_iter": [50, 100, 150, 200, 300, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_leaf_nodes": [7, 15, 31, 63],
    "min_samples_leaf": [20, 30, 40, 50, 60],
    "l2_regularization": [0, 0.01, 0.1, 0.5, 1.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

hgb_search = RandomizedSearchCV(
    HistGradientBoostingClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=100,
    scoring="accuracy",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

hgb_search.fit(X_train, y_train)
hgb_best_model = hgb_search.best_estimator_

print("===== BEST HGB MODEL (5-fold CV) =====")
print(hgb_search.best_params_)
print("CV accuracy:", hgb_search.best_score_)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
===== BEST HGB MODEL (5-fold CV) =====
{'min_samples_leaf': 30, 'max_leaf_nodes': 31, 'max_iter': 100, 'learning_rate': 0.05, 'l2_regularization': 0.5}
CV accuracy: 0.8540832339463937


## XGBoost

In [43]:
param_dist = {
    "n_estimators": [100, 300, 500, 800],
    "learning_rate": [0.01, 0.03, 0.05, 0.07, 0.1],
    "max_depth": [2, 3, 4, 5, 6],
    "min_child_weight": [1, 3, 5, 7],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "reg_lambda": [0.5, 1, 2, 5, 10],
    "reg_alpha": [0, 0.1, 1, 2, 5]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric="logloss"),
    param_distributions=param_dist,
    n_iter=150,
    scoring="accuracy",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train)
xgb_best_model = xgb_search.best_estimator_

print("===== BEST XGB MODEL (5-fold CV) =====")
print(xgb_search.best_params_)
print("CV accuracy:", xgb_search.best_score_)

Fitting 5 folds for each of 150 candidates, totalling 750 fits
===== BEST XGB MODEL (5-fold CV) =====
{'subsample': 0.9, 'reg_lambda': 2, 'reg_alpha': 1, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.07, 'colsample_bytree': 0.7}
CV accuracy: 0.8552068294520117


## Majority vote

In [47]:
# ==============================
# Final Ensemble (RF + HGB + XGB) - hard voting
# ==============================

# Test predictions
rf_test_pred = rf_best_model.predict(X_test)
hgb_test_pred = hgb_best_model.predict(X_test)
xgb_test_pred = xgb_best_model.predict(X_test)

# Majority vote
vote_sum = (
    rf_test_pred +
    hgb_test_pred +
    xgb_test_pred
)

test_preds = (vote_sum >= 2).astype(int)

# Submission
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": test_preds
})

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


# Soft voting

In [48]:
rf_cv_acc = rf_search.best_score_
hgb_cv_acc = hgb_search.best_score_
xgb_cv_acc = xgb_search.best_score_

print(rf_cv_acc, hgb_cv_acc, xgb_cv_acc)

0.84509446990145 0.8540832339463937 0.8552068294520117


In [49]:
# ==============================
# Final Ensemble (RF + HGB + XGB) - soft voting
# ==============================

# Predicted probabilities (probability of Survived = 1) on the test set
rf_test_proba  = rf_best_model.predict_proba(X_test)[:, 1]
hgb_test_proba = hgb_best_model.predict_proba(X_test)[:, 1]
xgb_test_proba = xgb_best_model.predict_proba(X_test)[:, 1]

# Equal-weight average across three models
test_blend_proba = (
    rf_test_proba +
    hgb_test_proba +
    xgb_test_proba
) / 3

test_preds = (test_blend_proba >= 0.5).astype(int)

submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": test_preds
})

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


In [50]:
print(submission["Survived"].value_counts())

Survived
0    265
1    153
Name: count, dtype: int64


In [51]:
print(X_train.shape)
print(X_test.shape)

print(X_train.columns.equals(X_test.columns))

(891, 47)
(418, 47)
True


In [52]:
print(X_train.shape)
print(y_train.shape)

(891, 47)
(891,)


In [53]:
submission_check = pd.read_csv("/kaggle/working/submission.csv")
print(submission.equals(submission_check))

True
